# Capstone Project: Movie Recommendation Engine

Advanced Data Scientist Program - IIT Roorkee (iHUB DivyaSampark) with Intellipaat

Dataset: Netflix Prize dataset (combined_data_1.txt, movie_titles.csv)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9,5)


## Loading the ratings data

In [ ]:
raw = pd.read_csv("combined_data_1.txt", header=None, names=['cust_id','rating'], usecols=[0,1])
raw['rating'] = raw['rating'].astype('float32')
raw.head()


In [ ]:
nan_mask = raw['cust_id'].isnull()
nan_idx = raw.index[nan_mask].to_numpy()
movie_id_vals = raw.loc[nan_mask, 'cust_id'].str.replace(':', '', regex=False).astype('int32').to_numpy()

bounds = np.append(nan_idx, len(raw))
counts = np.diff(bounds) - 1
movie_id_col = np.repeat(movie_id_vals, counts)

ratings = raw[~nan_mask].reset_index(drop=True)
ratings['movie_id'] = movie_id_col
ratings['cust_id'] = ratings['cust_id'].astype('int32')
ratings['rating'] = ratings['rating'].astype('int8')

ratings.shape


In [ ]:
ratings.head(10)


In [ ]:
ratings['rating'].value_counts().sort_index()


## Loading movie titles

In [ ]:
titles = []
with open('movie_titles.csv', encoding='latin-1') as f:
    for line in f:
        parts = line.strip().split(',', 2)
        if len(parts) == 3:
            titles.append(parts)

movies = pd.DataFrame(titles, columns=['movie_id','year','title'])
movies['movie_id'] = movies['movie_id'].astype('int32')
movies.head()


## Objective 1: Most popular and highest rated movies

In [ ]:
movie_stats = ratings.groupby('movie_id').agg(num_ratings=('rating','count'), avg_rating=('rating','mean'))
movie_stats = movie_stats.merge(movies[['movie_id','title','year']], on='movie_id')
movie_stats = movie_stats.set_index('movie_id')
movie_stats.head()


In [ ]:
most_popular = movie_stats.sort_values('num_ratings', ascending=False).head(15)
most_popular[['title','year','num_ratings','avg_rating']]


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(y=most_popular['title'], x=most_popular['num_ratings'], color='#E50914')
plt.xlabel('Number of Ratings')
plt.ylabel('')
plt.title('Most Popular Movies by Rating Volume')
plt.tight_layout()
plt.savefig('most_popular_movies.png', dpi=150)
plt.show()


In [ ]:
benchmark = movie_stats['num_ratings'].quantile(0.7)
qualified = movie_stats[movie_stats['num_ratings'] >= benchmark]
highest_rated = qualified.sort_values('avg_rating', ascending=False).head(15)
highest_rated[['title','year','num_ratings','avg_rating']]


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(y=highest_rated['title'], x=highest_rated['avg_rating'], color='#221f1f')
plt.xlabel('Average Rating')
plt.ylabel('')
plt.title('Highest Rated Movies (min. rating volume applied)')
plt.tight_layout()
plt.savefig('highest_rated_movies.png', dpi=150)
plt.show()


## Objective 2: Best and worst rated movies

In [ ]:
worst_rated = qualified.sort_values('avg_rating', ascending=True).head(15)
worst_rated[['title','year','num_ratings','avg_rating']]


In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,6))
sns.barplot(y=highest_rated['title'].head(10), x=highest_rated['avg_rating'].head(10), ax=axes[0], color='#2ca02c')
axes[0].set_title('Top 10 Best Rated')
axes[0].set_xlabel('Average Rating')
axes[0].set_ylabel('')

sns.barplot(y=worst_rated['title'].head(10), x=worst_rated['avg_rating'].head(10), ax=axes[1], color='#d62728')
axes[1].set_title('Top 10 Worst Rated')
axes[1].set_xlabel('Average Rating')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('best_worst_movies.png', dpi=150)
plt.show()


## Objective 3: Recommendation model

In [ ]:
movie_rating_count = ratings.groupby('movie_id')['rating'].count()
cust_rating_count = ratings.groupby('cust_id')['rating'].count()

movie_benchmark = movie_rating_count.quantile(0.95)
cust_benchmark = cust_rating_count.quantile(0.95)

drop_movies = movie_rating_count[movie_rating_count < movie_benchmark].index
drop_custs = cust_rating_count[cust_rating_count < cust_benchmark].index

trimmed = ratings[~ratings['movie_id'].isin(drop_movies) & ~ratings['cust_id'].isin(drop_custs)]
trimmed.shape


In [ ]:
pivot = trimmed.pivot_table(index='cust_id', columns='movie_id', values='rating')
pivot.shape


In [ ]:
corr_matrix = pivot.corr(min_periods=50)
corr_matrix.shape


In [ ]:
def similar_movies(movie_id, min_count=50):
    corr = corr_matrix[movie_id].dropna()
    corr = pd.DataFrame(corr)
    corr.columns = ['correlation']
    corr = corr.join(movie_stats['num_ratings'])
    corr = corr[corr['num_ratings'] >= min_count]
    corr = corr.join(movies.set_index('movie_id')['title'])
    return corr.sort_values('correlation', ascending=False)


In [ ]:
sample_movie_id = corr_matrix.columns[0]
print(movies.set_index('movie_id').loc[sample_movie_id, 'title'])
similar_movies(sample_movie_id).head(10)


In [ ]:
def recommend_for_customer(cust_id, top_n=5):
    if cust_id not in pivot.index:
        return highest_rated[['title','avg_rating']].head(top_n)

    customer_ratings = pivot.loc[cust_id].dropna().sort_values(ascending=False)
    if len(customer_ratings) == 0:
        return highest_rated[['title','avg_rating']].head(top_n)

    seed_movie = customer_ratings.index[0]
    candidates = similar_movies(seed_movie)
    already_rated = customer_ratings.index
    candidates = candidates[~candidates.index.isin(already_rated)]
    return candidates[['title','correlation']].head(top_n)


In [ ]:
sample_customers = trimmed['cust_id'].drop_duplicates().sample(3, random_state=7)
for cid in sample_customers:
    print(f'Customer {cid}')
    print(recommend_for_customer(cid))
    print()


In [ ]:
all_recs = []
for cid in pivot.index:
    recs = recommend_for_customer(cid, top_n=3)
    for title in recs['title'].values:
        all_recs.append({'cust_id': cid, 'recommended_movie': title})

recommendations = pd.DataFrame(all_recs)
recommendations.to_csv('customer_recommendations.csv', index=False)
recommendations.shape


In [ ]:
recommendations.head(10)
